# PathSnap — Notebook 04: model comparison (Milestone 4)
**Owner: B (GeniePlunder), with input from everyone.** Reads every `*_results.json` written by the 03_models_* notebooks
(MyDrive/PathSnap/m4_outputs/) and builds the comparison required by Deliverables §3.1 / §3.3.
Partitions: CV numbers = 5-fold trip-grouped CV on the 80 training trips; "student-test" = the 10 student-test trips.

In [ ]:
import os, sys, json, glob, zipfile
import numpy as np, pandas as pd, matplotlib.pyplot as plt
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/PathSnap"
    M4, DS_DIR, ZIP_PATH, EXTRACT_DIR = f"{DRIVE}/m4_outputs", f"{DRIVE}/m3_outputs/dataset", f"{DRIVE}/Student folder.zip", "/content/pathsnap_data"
    REPO = "/content/01_pathsnap"
    if not os.path.isdir(REPO): os.system(f"git clone -q https://github.com/paya5am/01_pathsnap {REPO}")
    PKG_PARENT = REPO if os.path.isdir(f"{REPO}/pathsnap") else DRIVE
else:
    M4 = os.environ.get("PATHSNAP_M4", "m4_outputs"); DS_DIR = os.environ.get("PATHSNAP_DS", "m3_outputs/dataset")
    ZIP_PATH = os.environ.get("PATHSNAP_ZIP", "Student folder.zip"); EXTRACT_DIR = os.environ.get("PATHSNAP_EXTRACT", "pathsnap_data")
    PKG_PARENT = os.environ.get("PATHSNAP_PKG", ".")
sys.path.insert(0, PKG_PARENT)
if not os.path.isdir(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(EXTRACT_DIR)
DATA_ROOT = next(dp for dp, dn, _ in os.walk(EXTRACT_DIR) if "train" in dn and "student_test" in dn)
from pathsnap.modeling import load_dataset
from pathsnap.io import read_trip, trip_dir
FIG = os.path.join(M4, "comparison"); os.makedirs(FIG, exist_ok=True)
try: display
except NameError: display = print
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

R = {json.load(open(f))["model"]: json.load(open(f)) for f in sorted(glob.glob(f"{M4}/*_results.json"))}
ORDER = [m for m in ["logreg", "gaussian_nb", "random_forest", "hist_gb", "extra_trees", "lightgbm", "mlp", "knn"] if m in R]
missing = [m for m in ["logreg", "gaussian_nb", "random_forest", "hist_gb", "extra_trees", "lightgbm", "mlp", "knn"] if m not in R]
print("models found:", ORDER, "| missing (run their owner notebook):", missing)

## 1. Consolidated comparison table (Deliverables §3.1)

In [ ]:
def cfg(r):
    return ", ".join(f"{k}={v}" for k, v in r["config"].items() if k not in ("scaler",))[:90]
rows = []
for m in ORDER:
    r = R[m]; cv = r["cv"]; st = r["student_test"]
    rows.append({"Model": m, "Algorithm family": r["family"], "Key configuration": cfg(r),
                 "CV macro F1": f"{cv['val_row_macro_f1']['mean']:.4f} ± {cv['val_row_macro_f1']['std']:.4f}",
                 "CV Top-1": f"{cv['val_top1_accuracy']['mean']:.4f} ± {cv['val_top1_accuracy']['std']:.4f}",
                 "Student-test Top-1": round(st["top1_accuracy"], 4), "Route continuity (test)": round(st["route_continuity"], 4),
                 "Student-test macro F1": round(st["row_macro_f1"], 4), "Log loss (test)": round(st["log_loss"], 4),
                 "Precision (test)": round(st["row_precision"], 4), "Recall (test)": round(st["row_recall"], 4),
                 "Route-arc F1 (test)": round(st["route_arc_f1"], 4),
                 "Train time (s)": round(r["train_time_s"], 1), "Inference (ms / 1000 pts)": round(r["inference_ms_per_1000_points"], 1)})
T = pd.DataFrame(rows); T.to_csv(f"{FIG}/model_comparison_table.csv", index=False); display(T)

# numeric version for plots
N = pd.DataFrame([{"model": m,
                   **{f"cv_{k}": R[m]["cv"][f"val_{k}"]["mean"] for k in ["top1_accuracy", "row_macro_f1", "log_loss", "route_continuity"]},
                   **{f"cv_{k}_std": R[m]["cv"][f"val_{k}"]["std"] for k in ["top1_accuracy", "row_macro_f1", "log_loss"]},
                   **{f"test_{k}": R[m]["student_test"][k] for k in ["top1_accuracy", "row_macro_f1", "log_loss", "route_continuity", "route_arc_f1"]},
                   "train_top1": R[m]["cv"]["train_top1"]["mean"], "train_log_loss": R[m]["cv"]["train_log_loss"]["mean"],
                   "train_s": R[m]["train_time_s"], "infer_ms": R[m]["inference_ms_per_1000_points"]} for m in ORDER])
N.to_csv(f"{FIG}/model_comparison_numeric.csv", index=False)

## 2. Visual comparison (Deliverables §3.3) — every number comes from the result files above

In [ ]:
x = np.arange(len(N)); col = plt.cm.tab10(np.arange(len(N)))
fig, ax = plt.subplots(2, 2, figsize=(15, 9))
ax[0, 0].bar(x, N.cv_row_macro_f1, yerr=N.cv_row_macro_f1_std, color=col, capsize=4)
ax[0, 0].set(title="Cross-validation macro F1 (mean ± std over 5 trip-grouped folds)", ylim=(0.7, 1.0))
w = .38
ax[0, 1].bar(x - w / 2, N.test_top1_accuracy, w, label="Top-1 accuracy (points)"); ax[0, 1].bar(x + w / 2, N.test_row_macro_f1, w, label="macro F1 (rows)")
ax[0, 1].axhline(0.835, ls="--", c="k", lw=1, label="rule baseline: nearest arc pointing the car's way (0.835)")
ax[0, 1].set(title="Student-test: Top-1 and macro F1", ylim=(0.7, 1.06)); ax[0, 1].legend(fontsize=7, loc="upper left", ncol=2)
ax[1, 0].bar(x, N.test_log_loss, color=col); ax[1, 0].set(title="Student-test log loss (lower = better probabilities)")
ax[1, 1].scatter(N.infer_ms, N.cv_top1_accuracy, s=80, c=col)
for i, m in enumerate(N.model): ax[1, 1].annotate(m, (N.infer_ms[i], N.cv_top1_accuracy[i]), textcoords="offset points", xytext=(5, 4), fontsize=8)
ax[1, 1].set(xscale="log", xlabel="inference time (ms per 1000 GPS points, log)", ylabel="CV Top-1", title="Accuracy vs inference cost")
for a in ax.ravel()[:3]: a.set_xticks(x); a.set_xticklabels(N.model, rotation=30, ha="right")
plt.tight_layout(); plt.savefig(f"{FIG}/01_model_comparison.png", dpi=130); plt.show()

# fold stability
folds = pd.concat([pd.read_csv(f"{M4}/{m}_folds.csv").assign(model=m) for m in ORDER])
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for a, c, t in zip(ax, ["val_top1_accuracy", "val_row_macro_f1", "val_log_loss"], ["Top-1 per fold", "macro F1 per fold", "log loss per fold"]):
    for i, m in enumerate(ORDER):
        v = folds[folds.model == m][c].values; a.plot([i] * len(v), v, "o", alpha=.6, color=col[i]); a.plot(i, v.mean(), "k_", ms=18)
    a.set_xticks(range(len(ORDER))); a.set_xticklabels(ORDER, rotation=30, ha="right"); a.set_title(f"CV stability: {t}")
plt.tight_layout(); plt.savefig(f"{FIG}/02_fold_stability.png", dpi=130); plt.show()

# overfitting: train-fold vs validation
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].bar(x - w / 2, N.train_top1, w, label="train folds"); ax[0].bar(x + w / 2, N.cv_top1_accuracy, w, label="validation folds")
ax[0].set(title="Top-1: training vs validation (overfitting check)", ylim=(0.7, 1.0)); ax[0].legend()
ax[1].bar(x - w / 2, N.train_log_loss, w, label="train folds"); ax[1].bar(x + w / 2, N.cv_log_loss, w, label="validation folds")
ax[1].set(title="Log loss: training vs validation (KNN train ≈ 0: it memorises its own points)"); ax[1].legend()
for a in ax: a.set_xticks(x); a.set_xticklabels(N.model, rotation=30, ha="right")
plt.tight_layout(); plt.savefig(f"{FIG}/03_overfitting_check.png", dpi=130); plt.show()

# confusion matrices (student-test rows) + recorded training histories
fig, axs = plt.subplots(2, 4, figsize=(16, 7))
for a, m in zip(axs.ravel(), ORDER):
    cm = np.array(R[m]["student_test"]["confusion_matrix"]); cmn = cm / cm.sum(1, keepdims=True)
    a.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for i in range(2):
        for j in range(2): a.text(j, i, f"{cmn[i, j]:.2f}\n({cm[i, j]:,})", ha="center", va="center", fontsize=8)
    a.set(xticks=[0, 1], yticks=[0, 1], xticklabels=["pred 0", "pred 1"], yticklabels=["true 0", "true 1"], title=m)
for a in axs.ravel()[len(ORDER):]: a.axis("off")
plt.suptitle("Student-test confusion matrices (row-normalised)"); plt.tight_layout(); plt.savefig(f"{FIG}/04_confusion_matrices.png", dpi=130); plt.show()
hist = {m: R[m]["history"] for m in ORDER if R[m].get("history")}
if hist:
    fig, ax = plt.subplots(1, len(hist), figsize=(6 * len(hist), 4)); ax = np.atleast_1d(ax)
    for a, (m, h) in zip(ax, hist.items()):
        a2 = None
        for k, v in h.items():
            if "accuracy" in k:            # different scale → right-hand axis
                a2 = a2 or a.twinx(); a2.plot(v, color="tab:orange", label=k); a2.set_ylabel(k)
            else:
                a.plot(v, label=k)
        a.set(title=f"{m}: recorded training history", xlabel="epoch / iteration"); a.legend(fontsize=8, loc="center right")
        if a2: a2.legend(fontsize=8, loc="upper right")
    plt.tight_layout(); plt.savefig(f"{FIG}/05_training_histories.png", dpi=130); plt.show()
print("Training curves are shown only for models that record them:", list(hist))

## 3. Error analysis — what kind of mistakes do the models make? (student-test, labelled points)
For every wrongly matched point we check how the predicted arc relates to the reference:
* **twin** — the same road in the opposite direction (reverse arc);
* **adjacent on route** — the arc just before/after the reference on the reference route (the point is at the
  junction between two consecutive arcs; a timing/granularity error, mostly harmless for the final route);
* **connected** — shares a node with the reference arc (a turn-off at the same junction);
* **other** — anything else (parallel road, wrong road altogether);
* **miss** — the reference arc was not among the candidates (no model can get these).

In [ ]:
F, P = load_dataset(DS_DIR)
pts = P[(P.partition == "student_test") & (P.label_status == 0)]
cache = {}
def trip_info(tid):
    if tid not in cache:
        raw = read_trip(trip_dir(DATA_ROOT, "student_test", tid)); cache[tid] = (raw.arcs, raw.route)
    return cache[tid]
err_rows = []
for m in ORDER:
    pp = pd.read_csv(f"{M4}/{m}_student_test_point_preds.csv")
    j = pts.merge(pp[["trip_id", "pt", "pred_arc"]], on=["trip_id", "pt"])
    kinds = []
    for r in j.itertuples():
        if r.pred_arc == r.ref_arc: kinds.append("correct"); continue
        if not r.ref_in_cands: kinds.append("miss"); continue
        arcs, route = trip_info(r.trip_id); u, v = arcs[r.ref_arc]; pu, pv = arcs[r.pred_arc]
        nb = {route[k] for k in (r.route_pos - 1, r.route_pos + 1) if 0 <= k < len(route)}
        if (pu, pv) == (v, u): kinds.append("twin")
        elif r.pred_arc in nb or any((arcs[a] == arcs[r.pred_arc]).all() for a in nb): kinds.append("adjacent on route")
        elif len({pu, pv} & {u, v}): kinds.append("connected")
        else: kinds.append("other")
    vc = pd.Series(kinds).value_counts(normalize=True)
    err_rows.append({"model": m, **{k: vc.get(k, 0.0) for k in ["correct", "adjacent on route", "twin", "connected", "other", "miss"]}})
ERR = pd.DataFrame(err_rows).set_index("model"); ERR.to_csv(f"{FIG}/error_types_student_test.csv"); display(ERR.round(4))
ax = ERR.drop(columns="correct").plot.bar(stacked=True, figsize=(11, 4), title="Student-test: share of labelled points by error type (correct omitted)")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left"); plt.ylabel("share of labelled points"); plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.savefig(f"{FIG}/06_error_types.png", dpi=130); plt.show()
print("Top-1 if 'adjacent on route' were counted correct (diagnostic only, NOT our metric):",
      (ERR["correct"] + ERR["adjacent on route"]).round(4).to_dict())

## 4. Where are the errors? Moving vs stopped
Twin errors (right road, wrong direction) dominate. Hypothesis: they happen when the car is too slow for a heading
(`heading_known` = 0), because a single stopped GPS dot sits equally on both directions of a two-way road.

In [ ]:
ctx = F[F.partition == "student_test"].drop_duplicates(["trip_id", "pt"])[["trip_id", "pt", "heading_known", "speed"]]
mv_rows = []
for m in ORDER:
    pp = pd.read_csv(f"{M4}/{m}_student_test_point_preds.csv")
    j = pts.merge(pp[["trip_id", "pt", "pred_arc"]], on=["trip_id", "pt"]).merge(ctx, on=["trip_id", "pt"])
    ok = j.pred_arc == j.ref_arc
    tw = np.array([(r.pred_arc != r.ref_arc) and tuple(trip_info(r.trip_id)[0][r.pred_arc]) == tuple(trip_info(r.trip_id)[0][r.ref_arc][::-1])
                   for r in j.itertuples()])
    mv_rows.append({"model": m, "top1_moving": ok[j.heading_known == 1].mean(), "top1_stopped_or_slow": ok[j.heading_known == 0].mean(),
                    "share_points_stopped": 1 - j.heading_known.mean(), "twin_errors": int(tw.sum()),
                    "share_twin_errors_when_stopped": 1 - j.heading_known[tw].mean() if tw.any() else np.nan,
                    "median_speed_at_twin_errors": float(j.speed[tw].median()) if tw.any() else np.nan})
MV = pd.DataFrame(mv_rows).set_index("model"); MV.to_csv(f"{FIG}/moving_vs_stopped.csv"); display(MV.round(4))
ax = MV[["top1_moving", "top1_stopped_or_slow"]].plot.bar(figsize=(11, 4), ylim=(0, 1), title="Student-test Top-1: car moving (heading known) vs stopped/slow")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
ax.set_xticklabels(MV.index, rotation=30, ha="right"); plt.tight_layout(); plt.savefig(f"{FIG}/07_moving_vs_stopped.png", dpi=130); plt.show()
print("→ A stopped dot can't tell the two directions of a road apart. Only the sequence (where the car came from) can:"
      " this is the main job of the HMM in M7.")